# Datawell Consultancy
## Project: Financial Transaction Analytics
---
**Prepared by:** Datawell Consultancy
**Document:** Phase 1 — Data Quality & Profiling  
**Version:** 1.0

---

## Problem Statement

The client — a mid-size fintech company — has three core data sources: customer profiles, card records, and transaction logs. These datasets have never been formally audited or connected. The business cannot answer basic questions such as:

- What is our daily transaction success rate?
- Which customer segments drive the most revenue?
- Where are our highest-risk transactions occurring?

**Our engagement objective:** Profile, clean, and connect all three data sources — then build a unified analytics layer that answers these business questions.

---

## Dataset Overview

| File | Description | Key Fields |
|------|-------------|------------|
| `users_data.csv` | Customer demographics and financial profile | age, income, credit score, debt |
| `cards_data.csv` | Card details per customer | card brand, type, credit limit, chip |
| `transactions_data.csv` | Transaction records | date, amount, merchant, errors |
| `mcc_codes.json` | Merchant Category Code descriptions | mcc → category name |

## Step 0: Environment Setup

In [1]:
import os
import pandas as pd
import numpy as np
import duckdb
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
warnings.filterwarnings('ignore')

# Set base directory to repo root
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
RAW_DATA_DIR = os.path.join(BASE_DIR,'data','raw')
CLEANED_DATA_DIR = os.path.join(BASE_DIR,'data','cleaned')
NOTEBOOKS_DIR = os.path.join(BASE_DIR,'notebooks')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Chart style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')

print('Environment ready')

Environment ready


## Step 1: Load Raw Data

In [2]:
# Load all datasets
users_raw   = pd.read_csv(os.path.join(RAW_DATA_DIR,'users_data.csv'))
cards_raw   = pd.read_csv(os.path.join(RAW_DATA_DIR,'cards_data.csv'))
txns_raw    = pd.read_csv(os.path.join(RAW_DATA_DIR,'transactions_data.csv'),skiprows=lambda i: i > 0 and random.random() > 0.1)

with open(os.path.join(RAW_DATA_DIR,'mcc_codes.json'), 'r') as f:
    mcc_codes = json.load(f)

# Convert MCC codes to dataframe
mcc_df = pd.DataFrame(list(mcc_codes.items()), columns=['mcc', 'merchant_category'])
mcc_df['mcc'] = mcc_df['mcc'].astype(int)

print('All files loaded successfully')
print(f'   Users        : {users_raw.shape[0]:,} rows x {users_raw.shape[1]} columns')
print(f'   Cards        : {cards_raw.shape[0]:,} rows x {cards_raw.shape[1]} columns')
print(f'   Transactions : {txns_raw.shape[0]:,} rows x {txns_raw.shape[1]} columns')
print(f'   MCC Codes    : {len(mcc_codes)} categories')

All files loaded successfully
   Users        : 2,000 rows x 14 columns
   Cards        : 6,146 rows x 13 columns
   Transactions : 1,330,895 rows x 12 columns
   MCC Codes    : 109 categories


## Step 2: Data Quality Audit

We perform a structured data quality audit across four dimensions:
1. **Completeness** — Are there missing values?
2. **Uniqueness** — Are there duplicate records?
3. **Validity** — Are data types and formats correct?
4. **Consistency** — Do values make business sense?

In [4]:
def data_quality_report(df, name):
    """Generate a structured data quality report for any dataframe."""
    print(f'\n{"="*60}')
    print(f'  DATA QUALITY REPORT — {name.upper()}')
    print(f'{"="*60}')
    
    total_rows = len(df)
    total_cols = len(df.columns)
    
    print(f'\nShape      : {total_rows:,} rows x {total_cols} columns')
    print(f'Duplicates : {df.duplicated().sum():,} rows ({df.duplicated().sum()/total_rows*100:.1f}%)')
    
    # Null analysis
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / total_rows * 100).round(2)
    
    quality_df = pd.DataFrame({
        'Column'        : df.columns,
        'Data Type'     : df.dtypes.values,
        'Null Count'    : null_counts.values,
        'Null %'        : null_pct.values,
        'Unique Values' : [df[col].nunique() for col in df.columns]
    })
    
    print(f'\nColumn Profile:')
    print(quality_df.to_string(index=False))
    
    return quality_df

# Run audit on all three datasets
users_quality = data_quality_report(users_raw, 'Users Data')
cards_quality = data_quality_report(cards_raw, 'Cards Data')
txns_quality  = data_quality_report(txns_raw,  'Transactions Data')


  DATA QUALITY REPORT — USERS DATA

Shape      : 2,000 rows x 14 columns
Duplicates : 0 rows (0.0%)

Column Profile:
           Column Data Type  Null Count  Null %  Unique Values
               id     int64           0    0.00           2000
      current_age     int64           0    0.00             80
   retirement_age     int64           0    0.00             29
       birth_year     int64           0    0.00             80
      birth_month     int64           0    0.00             12
           gender    object           0    0.00              2
          address    object           0    0.00           1999
         latitude   float64           0    0.00            989
        longitude   float64           0    0.00           1224
per_capita_income    object           0    0.00           1754
    yearly_income    object           0    0.00           1948
       total_debt    object           0    0.00           1880
     credit_score     int64           0    0.00            321


## Step 3: Validity Checks — Business Rules

In [11]:
print('BUSINESS RULES VALIDATION')
print('='*60)

# USERS — business rule checks
print('\n USERS TABLE:')
print(f'   Age < 18 (invalid minors)     : {(users_raw["current_age"] < 18).sum()}')
print(f'   Age > 100 (unrealistic)       : {(users_raw["current_age"] > 100).sum()}')
print(f'   Negative yearly income        : {(users_raw["yearly_income"].str.replace("[$,]", "", regex=True).astype(float) < 0).sum()}')
print(f'   Retirement age < current age  : {(users_raw["retirement_age"] < users_raw["current_age"]).sum()}')
print(f'   Credit score < 300            : {(users_raw["credit_score"] < 300).sum()}')
print(f'   Credit score > 850            : {(users_raw["credit_score"] > 850).sum()}')

# CARDS — business rule checks
print('\n CARDS TABLE:')
print(f'   Cards on dark web             : {(cards_raw["card_on_dark_web"] == "Yes").sum()}')
print(f'   No chip cards                 : {(cards_raw["has_chip"] == "NO").sum()}')
print(f'   Zero credit limit             : {(cards_raw["credit_limit"].str.replace("[$,]", "", regex=True).astype(float) == 0).sum()}')

# TRANSACTIONS — business rule checks  
print('\n TRANSACTIONS TABLE:')
print(f'   Negative amounts (reversals)  : {(txns_raw["amount"].str.replace("[$,]", "", regex=True).astype(float) < 0).sum()}')
print(f'   Zero amount transactions      : {(txns_raw["amount"].str.replace("[$,]", "", regex=True).astype(float) == 0).sum()}')
print(f'   Transactions with errors      : {txns_raw["errors"].notna().sum():,}')
print(f'   Online transactions (no city) : {(txns_raw["merchant_city"] == "ONLINE").sum():,}')
print(f'   Unique error types            :')
if txns_raw['errors'].notna().any():
    for err, cnt in txns_raw['errors'].value_counts().items():
        print(f'      → {err}: {cnt:,}')

BUSINESS RULES VALIDATION

 USERS TABLE:
   Age < 18 (invalid minors)     : 0
   Age > 100 (unrealistic)       : 1
   Negative yearly income        : 0
   Retirement age < current age  : 277
   Credit score < 300            : 0
   Credit score > 850            : 0

 CARDS TABLE:
   Cards on dark web             : 0
   No chip cards                 : 646
   Zero credit limit             : 31

 TRANSACTIONS TABLE:
   Negative amounts (reversals)  : 65909
   Zero amount transactions      : 1061
   Transactions with errors      : 21,125
   Online transactions (no city) : 157,082
   Unique error types            :
      → Insufficient Balance: 13,095
      → Bad PIN: 3,140
      → Technical Glitch: 2,686
      → Bad Card Number: 735
      → Bad Expiration: 640
      → Bad CVV: 627
      → Bad Zipcode: 105
      → Bad PIN,Insufficient Balance: 30
      → Insufficient Balance,Technical Glitch: 25
      → Bad Card Number,Insufficient Balance: 8
      → Bad Expiration,Insufficient Balance: 6
  

## Step 4: Data Cleaning

Based on the quality audit above, we apply the following cleaning logic to each table.

In [12]:
# ─────────────────────────────────────────
# CLEAN USERS TABLE
# ─────────────────────────────────────────
users = users_raw.copy()

# Fix currency columns — remove $ and commas
currency_cols = ['per_capita_income', 'yearly_income', 'total_debt']
for col in currency_cols:
    users[col] = users[col].str.replace('[$,]', '', regex=True).astype(float)

# Create age segments
users['age_group'] = pd.cut(
    users['current_age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
)

# Create credit score segments
users['credit_tier'] = pd.cut(
    users['credit_score'],
    bins=[0, 579, 669, 739, 799, 850],
    labels=['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional']
)

# Create income segments
users['income_band'] = pd.cut(
    users['yearly_income'],
    bins=[0, 30000, 60000, 100000, 200000, float('inf')],
    labels=['Low', 'Lower-Mid', 'Mid', 'Upper-Mid', 'High']
)

# Debt to income ratio
users['debt_to_income_ratio'] = (users['total_debt'] / users['yearly_income']).round(2)

# Years to retirement
users['years_to_retirement'] = users['retirement_age'] - users['current_age']

print(f' Users cleaned: {len(users):,} records')
print(f'   New columns added: age_group, credit_tier, income_band, debt_to_income_ratio, years_to_retirement')
users.head(3)

 Users cleaned: 2,000 records
   New columns added: age_group, credit_tier, income_band, debt_to_income_ratio, years_to_retirement


,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,age_group,credit_tier,income_band,debt_to_income_ratio,years_to_retirement
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,"29,278.00","59,696.00","127,613.00",787,5,46-55,Very Good,Lower-Mid,2.14,13
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,"37,891.00","77,254.00","191,349.00",701,5,46-55,Good,Mid,2.48,15
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,"22,681.00","33,483.00",196.00,698,5,65+,Good,Lower-Mid,0.01,-14


In [13]:
# ─────────────────────────────────────────
# CLEAN CARDS TABLE
# ─────────────────────────────────────────
cards = cards_raw.copy()

# Fix credit limit — remove $ and commas
cards['credit_limit'] = cards['credit_limit'].str.replace('[$,]', '', regex=True).astype(float)

# Parse expiry date
cards['expires'] = pd.to_datetime(cards['expires'], format='%m/%Y')
cards['acct_open_date'] = pd.to_datetime(cards['acct_open_date'], format='%m/%Y')

# Card age in years
cards['card_age_years'] = ((pd.Timestamp.now() - cards['acct_open_date']).dt.days / 365).round(1)

# Is card expired?
cards['is_expired'] = cards['expires'] < pd.Timestamp.now()

# Risk flag — card on dark web
cards['is_high_risk'] = cards['card_on_dark_web'] == 'Yes'

# Credit limit tiers
cards['limit_tier'] = pd.cut(
    cards['credit_limit'],
    bins=[0, 5000, 15000, 30000, 60000, float('inf')],
    labels=['Very Low', 'Low', 'Mid', 'High', 'Premium']
)

print(f' Cards cleaned: {len(cards):,} records')
print(f' Expired cards    : {cards["is_expired"].sum():,}')
print(f' High risk cards  : {cards["is_high_risk"].sum():,}')
cards.head(3)

 Cards cleaned: 6,146 records
 Expired cards    : 6,146
 High risk cards  : 0


,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,card_age_years,is_expired,is_high_risk,limit_tier
0,4524,825,Visa,Debit,4344676511950444,2022-12-01,623,YES,2,"24,295.00",2002-09-01,2008,No,23.70,True,False,Mid
1,2731,825,Visa,Debit,4956965974959986,2020-12-01,393,YES,2,"21,968.00",2014-04-01,2014,No,12.10,True,False,Mid
2,3701,825,Visa,Debit,4582313478255491,2024-02-01,719,YES,2,"46,414.00",2003-07-01,2004,No,22.90,True,False,High


In [14]:
# ─────────────────────────────────────────
# CLEAN TRANSACTIONS TABLE
# ─────────────────────────────────────────
txns = txns_raw.copy()

# Parse date
txns['date'] = pd.to_datetime(txns['date'])

# Fix amount — remove $ and handle negatives
txns['amount'] = txns['amount'].str.replace('[$,]', '', regex=True).astype(float)

# Flag reversals (negative amounts)
txns['is_reversal'] = txns['amount'] < 0
txns['amount_abs'] = txns['amount'].abs()

# Flag errors
txns['has_error'] = txns['errors'].notna()
txns['error_type'] = txns['errors'].fillna('No Error')

# Transaction channel
txns['channel'] = txns['use_chip'].map({
    'Swipe Transaction'  : 'In-Store Swipe',
    'Online Transaction' : 'Online',
    'Chip Transaction'   : 'In-Store Chip'
}).fillna('Other')

# Is online?
txns['is_online'] = txns['merchant_city'] == 'ONLINE'

# Date parts
txns['txn_year']    = txns['date'].dt.year
txns['txn_month']   = txns['date'].dt.month
txns['txn_day']     = txns['date'].dt.day
txns['txn_hour']    = txns['date'].dt.hour
txns['txn_dow']     = txns['date'].dt.day_name()
txns['txn_date']    = txns['date'].dt.date

# Merge MCC codes
txns = txns.merge(mcc_df, on='mcc', how='left')
txns['merchant_category'] = txns['merchant_category'].fillna('Unknown Category')

# Amount buckets
txns['amount_bucket'] = pd.cut(
    txns['amount_abs'],
    bins=[0, 20, 50, 100, 250, 500, float('inf')],
    labels=['Micro (<$20)', 'Small ($20-50)', 'Medium ($50-100)', 
            'Large ($100-250)', 'Major ($250-500)', 'Premium ($500+)']
)

print(f'   Transactions cleaned: {len(txns):,} records')
print(f'   Date range      : {txns["date"].min().date()} to {txns["date"].max().date()}')
print(f'   Reversals       : {txns["is_reversal"].sum():,} ({txns["is_reversal"].mean()*100:.1f}%)')
print(f'   Error txns      : {txns["has_error"].sum():,} ({txns["has_error"].mean()*100:.1f}%)')
print(f'   Online txns     : {txns["is_online"].sum():,} ({txns["is_online"].mean()*100:.1f}%)')
txns.head(3)

   Transactions cleaned: 1,330,895 records
   Date range      : 2010-01-01 to 2019-10-31
   Reversals       : 65,909 (5.0%)
   Error txns      : 21,125 (1.6%)
   Online txns     : 157,082 (11.8%)


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,is_reversal,amount_abs,has_error,error_type,channel,is_online,txn_year,txn_month,txn_day,txn_hour,txn_dow,txn_date,merchant_category,amount_bucket
0,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,"20,776.00",5813,NaN,False,46.41,False,No Error,In-Store Swipe,False,2010,1,1,0,Friday,2010-01-01,Drinking Places (Alcoholic Beverages),Small ($20-50)
1,7475359,2010-01-01 00:48:00,1127,3869,22.57,Online Transaction,39021,ONLINE,NaN,NaN,4784,NaN,False,22.57,False,No Error,Online,True,2010,1,1,0,Friday,2010-01-01,Tolls and Bridge Fees,Small ($20-50)
2,7475363,2010-01-01 00:57:00,285,3442,7.80,Swipe Transaction,81536,Watsonville,CA,"95,076.00",5310,NaN,False,7.80,False,No Error,In-Store Swipe,False,2010,1,1,0,Friday,2010-01-01,Discount Stores,Micro (<$20)


## Step 5: Build Unified Master Dataset

Join all three tables into a single unified analytics layer.

In [15]:
# Join using DuckDB SQL
master = duckdb.query("""
    SELECT 
        -- Transaction fields
        t.id                   AS txn_id,
        t.date                 AS txn_datetime,
        t.txn_date,
        t.txn_year,
        t.txn_month,
        t.txn_hour,
        t.txn_dow,
        t.amount,
        t.amount_abs,
        t.amount_bucket,
        t.channel,
        t.is_online,
        t.is_reversal,
        t.has_error,
        t.error_type,
        t.merchant_id,
        t.merchant_city,
        t.merchant_state,
        t.mcc,
        t.merchant_category,

        -- Card fields
        c.id                   AS card_id,
        c.card_brand,
        c.card_type,
        c.has_chip,
        c.credit_limit,
        c.limit_tier,
        c.is_expired,
        c.is_high_risk,
        c.card_age_years,

        -- User fields
        u.id                   AS user_id,
        u.current_age,
        u.age_group,
        u.gender,
        u.yearly_income,
        u.income_band,
        u.total_debt,
        u.credit_score,
        u.credit_tier,
        u.debt_to_income_ratio,
        u.num_credit_cards

    FROM txns t
    LEFT JOIN cards c ON t.card_id = c.id
    LEFT JOIN users u ON t.client_id = u.id
""").df()

print(f'   Master dataset built: {len(master):,} rows x {len(master.columns)} columns')
print(f'   Join coverage — cards : {master["card_id"].notna().mean()*100:.1f}%')
print(f'   Join coverage — users : {master["user_id"].notna().mean()*100:.1f}%')
master.head(3)

   Master dataset built: 1,330,895 rows x 40 columns
   Join coverage — cards : 100.0%
   Join coverage — users : 100.0%


,txn_id,txn_datetime,txn_date,txn_year,txn_month,txn_hour,txn_dow,amount,amount_abs,amount_bucket,channel,is_online,is_reversal,has_error,error_type,merchant_id,merchant_city,merchant_state,mcc,merchant_category,card_id,card_brand,card_type,has_chip,credit_limit,limit_tier,is_expired,is_high_risk,card_age_years,user_id,current_age,age_group,gender,yearly_income,income_band,total_debt,credit_score,credit_tier,debt_to_income_ratio,num_credit_cards
0,13690222,2013-12-08 07:15:00,2013-12-08,2013,12,7,Sunday,82.27,82.27,Medium ($50-100),In-Store Swipe,False,False,False,No Error,61195,Brighton,CO,5541,Service Stations,4783,Mastercard,Debit,YES,"25,900.00",Mid,True,False,15.70,303,94,65+,Male,"60,080.00",Mid,"1,807.00",690,Good,0.03,6
1,13690226,2013-12-08 07:15:00,2013-12-08,2013,12,7,Sunday,21.77,21.77,Small ($20-50),In-Store Swipe,False,False,False,No Error,41080,Charleston,SC,5411,"Grocery Stores, Supermarkets",6095,Visa,Debit,YES,"15,194.00",Mid,True,False,12.50,1088,49,46-55,Female,"34,982.00",Lower-Mid,"134,631.00",765,Very Good,3.85,3
2,13690228,2013-12-08 07:15:00,2013-12-08,2013,12,7,Sunday,80.00,80.00,Medium ($50-100),In-Store Swipe,False,False,False,No Error,27092,Minneapolis,MN,4829,Money Transfer,2003,Discover,Credit,YES,"8,400.00",Low,True,False,29.20,1774,52,46-55,Male,"42,129.00",Lower-Mid,"101,742.00",758,Very Good,2.42,4


## Step 6: Export Cleaned Data

In [16]:
# Export all cleaned tables
users.to_csv(os.path.join(CLEANED_DATA_DIR,'users_cleaned.csv'), index=False)
cards.to_csv(os.path.join(CLEANED_DATA_DIR,'cards_cleaned.csv'), index=False)
txns.to_csv(os.path.join(CLEANED_DATA_DIR,'transactions_cleaned.csv'), index=False)
master.to_csv(os.path.join(CLEANED_DATA_DIR,'master_dataset.csv'), index=False)

print('    All cleaned files exported to data/cleaned/')
print(f'   users_cleaned.csv         : {len(users):,} rows')
print(f'   cards_cleaned.csv         : {len(cards):,} rows')
print(f'   transactions_cleaned.csv  : {len(txns):,} rows')
print(f'   master_dataset.csv        : {len(master):,} rows')

    All cleaned files exported to data/cleaned/
   users_cleaned.csv         : 2,000 rows
   cards_cleaned.csv         : 6,146 rows
   transactions_cleaned.csv  : 1,330,895 rows
   master_dataset.csv        : 1,330,895 rows
